# Letter Boxed solver (Ruby)

This notebook uses a real dictionary from `dwyl/english-words` and searches for a minimum-word solution. Among equal-length solutions, it prefers more normal-looking words.

In [ ]:
require 'set'

# ===============================================
# Letter Boxed solver with tie-breaking for
# more normal-looking words
# ===============================================

SIDES = [
  %w[a q e], # top
  %w[z s t], # right
  %w[u r c], # bottom
  %w[i n b]  # left
]

WORD_LIST_PATH = '../dictionary/words_alpha.txt'

MIN_WORD_LEN = 3
MAX_DEPTH = 6
SHOW_TOP = 10
MAX_WORD_LEN = 12
PREFER_SHORTER_TOTAL_LENGTH = true

ODD_SUFFIXES = %w[
  ia ism ist ists itis oidal ology ologies ation ations
  ization izations iveness ously ium iums
]

def uncommon_penalty(word)
  penalty = 0
  penalty += 100 if word.length > MAX_WORD_LEN
  penalty += 25 if word.count('jqxz') >= 3
  penalty += 18 if ODD_SUFFIXES.any? { |s| word.end_with?(s) }
  penalty += 14 if word.scan(/[aeiouy]/).length <= 1
  penalty += 10 if word.match?(/(.)\1\1/)
  penalty += 8 if word.match?(/[jqzx]{2}/)
  penalty += 5 if word.length >= 10
  penalty += 3 if word.length == 3
  penalty
end

Word = Struct.new(:text, :start_ch, :end_ch, :mask, :score)

class LetterBoxedSolver
  def initialize(sides, word_list_path)
    @sides = sides.map { |side| side.map(&:downcase) }
    @letters = @sides.flatten
    @letter_set = @letters.to_set
    @word_list_path = word_list_path

    raise "Word list not found: #{@word_list_path}" unless File.file?(@word_list_path)

    @char_to_side = {}
    @char_to_bit = {}

    @sides.each_with_index do |side, side_idx|
      side.each { |ch| @char_to_side[ch] = side_idx }
    end

    @letters.each_with_index do |ch, idx|
      @char_to_bit[ch] = idx
    end

    @goal_mask = (1 << @letters.length) - 1
  end

  def solve(show_top: SHOW_TOP)
    words = load_valid_words
    puts "Valid words after filtering: #{words.length}"

    by_start = Hash.new { |h, k| h[k] = [] }
    words.each { |w| by_start[w.start_ch] << w }

    1.upto(MAX_DEPTH) do |depth|
      puts "Searching depth #{depth}..."
      solutions = search_all_best(words, by_start, depth, show_top)
      next if solutions.empty?

      puts
      puts "Minimum solution uses #{depth} word(s):"
      puts

      solutions.each_with_index do |sol, i|
        puts "#{i + 1}. #{sol.map(&:text).join(' -> ')}"
      end

      return solutions.map { |sol| sol.map(&:text) }
    end

    puts "No solution found up to depth #{MAX_DEPTH}."
    nil
  end

  private

  def load_valid_words
    words = []
    seen = Set.new

    File.foreach(@word_list_path, chomp: true) do |line|
      word = line.strip.downcase
      next if seen.include?(word)
      next if word.length < MIN_WORD_LEN
      next unless word.match?(/\A[a-z]+\z/)
      next unless valid_word?(word)

      seen << word
      mask = word_mask(word)
      score = word_score(word, mask)
      words << Word.new(word, word[0], word[-1], mask, score)
    end

    grouped = {}
    words.each do |w|
      key = [w.start_ch, w.end_ch, w.mask]
      prev = grouped[key]
      grouped[key] = w if prev.nil? || w.score < prev.score
    end

    pruned = grouped.values
    pruned.sort_by! { |w| [w.score, w.text.length, w.text] }
    pruned
  end

  def valid_word?(word)
    chars = word.chars
    return false unless chars.all? { |ch| @letter_set.include?(ch) }

    chars.each_cons(2) do |a, b|
      return false if @char_to_side[a] == @char_to_side[b]
    end

    true
  end

  def word_mask(word)
    mask = 0
    word.each_char { |ch| mask |= (1 << @char_to_bit[ch]) }
    mask
  end

  def word_score(word, mask)
    unique_letters = bitcount(mask)
    score = 0
    score += uncommon_penalty(word)
    score -= unique_letters * 8
    score -= [word.length, 8].min
    score
  end

  def search_all_best(words, by_start, depth, limit)
    best_solutions = []
    best_rank = nil
    dead = {}

    words.each do |word|
      dfs(
        path: [word],
        used_mask: word.mask,
        depth_left: depth - 1,
        by_start: by_start,
        best_solutions: best_solutions,
        best_rank: -> { best_rank },
        set_best_rank: ->(r) { best_rank = r },
        limit: limit,
        dead: dead
      )
    end

    best_solutions.sort_by { |sol| solution_rank(sol) }.first(limit)
  end

  def dfs(path:, used_mask:, depth_left:, by_start:, best_solutions:, best_rank:, set_best_rank:, limit:, dead:)
    if used_mask == @goal_mask
      rank = solution_rank(path)
      current_best = best_rank.call

      if current_best.nil? || rank < current_best
        set_best_rank.call(rank)
        best_solutions.clear
        best_solutions << path.dup
      elsif rank == current_best
        best_solutions << path.dup if best_solutions.length < limit * 3
      end
      return
    end

    return if depth_left == 0

    state = [path[-1].end_ch, used_mask, depth_left]
    return if dead[state]

    candidates = by_start[path[-1].end_ch]
    candidates = candidates.sort_by do |w|
      gain = bitcount(w.mask & ~used_mask)
      [-gain, w.score, w.text.length, w.text]
    end

    candidates.each do |word|
      new_mask = used_mask | word.mask
      dfs(
        path: path + [word],
        used_mask: new_mask,
        depth_left: depth_left - 1,
        by_start: by_start,
        best_solutions: best_solutions,
        best_rank: best_rank,
        set_best_rank: set_best_rank,
        limit: limit,
        dead: dead
      )
    end

    dead[state] = true
  end

  def solution_rank(solution)
    total_score = solution.sum(&:score)
    total_length = solution.sum { |w| w.text.length }
    longest = solution.map { |w| w.text.length }.max
    words_text = solution.map(&:text).join('|')

    if PREFER_SHORTER_TOTAL_LENGTH
      [total_score, total_length, longest, words_text]
    else
      [total_score, longest, total_length, words_text]
    end
  end

  def bitcount(n)
    n.to_s(2).count('1')
  end
end

solver = LetterBoxedSolver.new(SIDES, WORD_LIST_PATH)
solver.solve
